<span style="color:pink; font-size:15px;">File to find the optimal unitaries for the LTO protocol using optimization techniques for gaining the maximum ergotropy advantage</span>

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from state_gen import generate_state
# from energy import energy, free_energy, passive_energy_g, passive_energy_l, ergotropy_gap, global_ergo, local_ergo
from local_to import sho_ham, gibbs_state, LTO_step
# from unitary import deg_unitary
from optimize import search_over_states, random_search, nelder_mead_search, lbfgs_search
from tqdm import tqdm
np.set_printoptions(linewidth=np.inf, precision=4, suppress=True)

def fmt(M): return np.array2string(M, precision=4, suppress_small=True, floatmode='fixed', max_line_width=80)

In [ ]:
# Default Parameters
beta_a=1.0
beta_b=0.8
bath_dim_a = bath_dim_b = 10
sys_dim_a = sys_dim_b = 4
omega_a = omega_b = 0.001
w1=0.001
w2=0.002

In [3]:
# n_states=50; n_unitaries=200
# search_over_states(n_states, n_unitaries, beta_a, beta_b, sys_dim_a, sys_dim_b, w1, w2, omega_a, omega_b, bath_dim_a, bath_dim_b)

In [ ]:
# Initialize Hamiltonians and Gibbs states
Hs1 = sho_ham(dim=sys_dim_a, omega=w1);          Hs2 = sho_ham(dim=sys_dim_b, omega=w2)
Hb1 = sho_ham(bath_dim_a, omega_a); Hb2 = sho_ham(bath_dim_b, omega_b)
gamma_Ra = gibbs_state(Hb1, beta_a)
gamma_Rb = gibbs_state(Hb2, beta_b)

# master tracker across all states
master_best = {
    'global': (-np.inf, None, None, None, None),  # (delta, rho12, sigma12, Ua, Ub)
    'local':  (-np.inf, None, None, None, None),
    'gap':    (-np.inf, None, None, None, None),
}

n_states    = 20
n_restarts  = 10
maxiter     = 500

for s in range(n_states):
    print(f"\n{'='*50}")
    print(f"  STATE {s+1}/{n_states}")
    print(f"{'='*50}")

    rho12 = generate_state(sys_dim_a * sys_dim_b)

    # run both optimizers
    best_nm   = nelder_mead_search(rho12, gamma_Ra, gamma_Rb, Hs1, Hs2, Hb1, Hb2, sys_dim_a, sys_dim_b, w1, w2, n_restarts, maxiter)
    best_lbfgs = lbfgs_search(rho12, gamma_Ra, gamma_Rb, Hs1, Hs2, Hb1, Hb2, sys_dim_a, sys_dim_b, w1, w2, n_restarts, maxiter)

    # update master tracker
    for target in ['global', 'local', 'gap']:
        for best, label in [(best_nm, 'NM'), (best_lbfgs, 'LBFGS')]:
            delta, sigma12, Ua, Ub = best[target]
            if delta > master_best[target][0]:
                master_best[target] = (delta, rho12.copy(), sigma12.copy(), Ua.copy(), Ub.copy())
                print(f"  ✓ New best Δ({target}) = {delta:.6f} [{label}]")

# final summary
print(f"\n{'='*50}")
print("  FINAL BEST ACROSS ALL STATES")
print(f"{'='*50}")
for target in ['global', 'local', 'gap']:
    delta = master_best[target][0]
    print(f"  Δ({target:6s}) = {delta:.6f}  "
          f"{'✓ > 0' if delta > 1e-8 else '✗ not found'}")
print(f"{'='*50}")


  STATE 1/20


Nelder-Mead:   0%|          | 0/10 [00:00<?, ?restart/s]

Nelder-Mead:  10%|█         | 1/10 [26:55<4:02:19, 1615.45s/restart]

  restart   1  Δglobal=-0.000204  Δlocal=-0.000034  Δgap=-0.000169


Nelder-Mead:  20%|██        | 2/10 [53:59<3:36:02, 1620.27s/restart]

...

  restart   9  Δglobal=-0.000204  Δlocal=-0.000033  Δgap=-0.000169


Nelder-Mead: 100%|██████████| 10/10 [5:09:00<00:00, 1854.05s/restart]


  restart  10  Δglobal=-0.000204  Δlocal=-0.000033  Δgap=-0.000169

  NELDER-MEAD RESULTS
  Best Δ(global) = -0.000204  ✗ not found
  Best Δ(local ) = -0.000033  ✗ not found
  Best Δ(gap   ) = -0.000169  ✗ not found


L-BFGS-B:   0%|          | 0/10 [07:36<?, ?restart/s]


KeyboardInterrupt: 